<a href="https://colab.research.google.com/github/maorufa/artemisinin-resistance-transcriptomics/blob/main/notebooks/GSE25878_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# Download GSE25878 for initial screening
!pip -q install GEOparse
import GEOparse
import pandas as pd
import numpy as np

gse25878 = GEOparse.get_GEO(
    "GSE25878",
    destdir="data/raw",
    how="brief"
)

print("DATASET: GSE25878")
print("=" * 45)

print("Total GEO samples:", len(gse25878.gsms))
print("Platform IDs:", list(gse25878.gpls.keys()))

# Inspect the first five samples
sample_records = []

for gsm_id, gsm in list(gse25878.gsms.items())[:5]:

    sample_records.append({
        "GSM_ID": gsm_id,
        "Title": gsm.metadata.get("title", [""])[0],
        "Characteristics": gsm.metadata.get(
            "characteristics_ch1", []
        )
    })

sample_preview = pd.DataFrame(sample_records)

display(sample_preview)

17-Nov-2026 12:37:06 DEBUG utils - Directory data/raw already exists. Skipping.
DEBUG:GEOparse:Directory data/raw already exists. Skipping.
17-Nov-2026 12:37:06 INFO GEOparse - File already exist: using local version.
INFO:GEOparse:File already exist: using local version.
17-Nov-2026 12:37:06 INFO GEOparse - Parsing data/raw/GSE25878.txt: 
INFO:GEOparse:Parsing data/raw/GSE25878.txt: 
17-Nov-2026 12:37:06 DEBUG GEOparse - SAMPLE: GSM635264
DEBUG:GEOparse:SAMPLE: GSM635264
17-Nov-2026 12:37:06 DEBUG GEOparse - SAMPLE: GSM635265
DEBUG:GEOparse:SAMPLE: GSM635265
17-Nov-2026 12:37:06 DEBUG GEOparse - SAMPLE: GSM635266
DEBUG:GEOparse:SAMPLE: GSM635266
17-Nov-2026 12:37:06 DEBUG GEOparse - SAMPLE: GSM635267
DEBUG:GEOparse:SAMPLE: GSM635267
17-Nov-2026 12:37:06 DEBUG GEOparse - SAMPLE: GSM635268
DEBUG:GEOparse:SAMPLE: GSM635268
17-Nov-2026 12:37:06 DEBUG GEOparse - SAMPLE: GSM635269
DEBUG:GEOparse:SAMPLE: GSM635269
17-Nov-2026 12:37:06 DEBUG GEOparse - SAMPLE: GSM635270
DEBUG:GEOparse:SAMPLE:

DATASET: GSE25878
Total GEO samples: 91
Platform IDs: []


,GSM_ID,Title,Characteristics
0,GSM635264,ex-vivo_ ART-sensitive_ Xepon Laos_BMT061_0H,"[culture type: Ex-vivo culture, time point: 0H..."
1,GSM635265,ex-vivo_ ART-sensitive_ Xepon Laos_BMT061_2H,"[culture type: Ex-vivo culture, time point: 2H..."
2,GSM635266,ex-vivo_ ART-sensitive_ Xepon Laos_BMT061_4H,"[culture type: Ex-vivo culture, time point: 4H..."
3,GSM635267,ex-vivo_ ART-sensitive_ Xepon Laos_BMT061_6H,"[culture type: Ex-vivo culture, time point: 6H..."
4,GSM635268,ex-vivo_ ART-sensitive_ Xepon Laos_BMT061_8H,"[culture type: Ex-vivo culture, time point: 8H..."


In [15]:
# Extract sample design information from all 91 samples

sample_records = []

for gsm_id, gsm in gse25878.gsms.items():

    title = gsm.metadata.get("title", [""])[0]

    characteristics = gsm.metadata.get(
        "characteristics_ch1", []
    )

    sample_records.append({
        "GSM_ID": gsm_id,
        "Title": title,
        "Characteristics": " | ".join(characteristics)
    })

gse25878_summary = pd.DataFrame(sample_records)

# Identify resistance labels
gse25878_summary["Resistance_Group"] = (
    gse25878_summary["Title"]
    .str.extract(r"(ART-sensitive|ART-resistant)", expand=False)
)

# Identify sampling timepoints
gse25878_summary["Timepoint"] = (
    gse25878_summary["Title"]
    .str.extract(r"_(\d+)H$", expand=False)
)

# Identify isolate IDs
gse25878_summary["Isolate_ID"] = (
    gse25878_summary["Title"]
    .str.extract(r"_(BMT\d+)_", expand=False)
)

print("Total samples:", len(gse25878_summary))

print("\nResistance groups:")
display(gse25878_summary["Resistance_Group"].value_counts(dropna=False))

print("\nUnique isolates:", gse25878_summary["Isolate_ID"].nunique())

print("\nSamples per timepoint:")
display(gse25878_summary["Timepoint"].value_counts(dropna=False).sort_index())

print("\nSamples per isolate and resistance group:")
display(
    gse25878_summary.groupby(
        ["Isolate_ID", "Resistance_Group"]
    ).size().to_frame("Number_of_samples")
)

print("\nSample preview:")
display(gse25878_summary.head(10))

Total samples: 91

Resistance groups:


,count
Resistance_Group,
ART-sensitive,64
ART-resistant,27



Unique isolates: 3

Samples per timepoint:


,count
Timepoint,
0,11
12,7
16,5
18,6
2,2
24,11
30,6
32,5
36,6



Samples per isolate and resistance group:


,,Number_of_samples
Isolate_ID,Resistance_Group,
BMT061,ART-sensitive,10
BMT076,ART-sensitive,9
BMT077,ART-sensitive,9



Sample preview:


,GSM_ID,Title,Characteristics,Resistance_Group,Timepoint,Isolate_ID
0,GSM635264,ex-vivo_ ART-sensitive_ Xepon Laos_BMT061_0H,culture type: Ex-vivo culture | time point: 0H...,ART-sensitive,0,BMT061
1,GSM635265,ex-vivo_ ART-sensitive_ Xepon Laos_BMT061_2H,culture type: Ex-vivo culture | time point: 2H...,ART-sensitive,2,BMT061
2,GSM635266,ex-vivo_ ART-sensitive_ Xepon Laos_BMT061_4H,culture type: Ex-vivo culture | time point: 4H...,ART-sensitive,4,BMT061
3,GSM635267,ex-vivo_ ART-sensitive_ Xepon Laos_BMT061_6H,culture type: Ex-vivo culture | time point: 6H...,ART-sensitive,6,BMT061
4,GSM635268,ex-vivo_ ART-sensitive_ Xepon Laos_BMT061_8H,culture type: Ex-vivo culture | time point: 8H...,ART-sensitive,8,BMT061
5,GSM635269,ex-vivo_ ART-sensitive_ Xepon Laos_BMT061_12H,culture type: Ex-vivo culture | time point: 12...,ART-sensitive,12,BMT061
6,GSM635270,ex-vivo_ ART-sensitive_ Xepon Laos_BMT061_16H,culture type: Ex-vivo culture | time point: 16...,ART-sensitive,16,BMT061
7,GSM635271,ex-vivo_ ART-sensitive_ Xepon Laos_BMT061_24H,culture type: Ex-vivo culture | time point: 24...,ART-sensitive,24,BMT061
8,GSM635272,ex-vivo_ ART-sensitive_ Xepon Laos_BMT061_32H,culture type: Ex-vivo culture | time point: 32...,ART-sensitive,32,BMT061
9,GSM635273,ex-vivo_ ART-sensitive_ Xepon Laos_BMT061_42H,culture type: Ex-vivo culture | time point: 42...,ART-sensitive,42,BMT061


In [16]:
# Check resistance groups represented by each isolate

print("Isolates by resistance group:")
display(
    pd.crosstab(
        gse25878_summary["Isolate_ID"],
        gse25878_summary["Resistance_Group"],
        margins=True
    )
)

print("\nTimepoints available for each isolate:")
display(
    pd.crosstab(
        gse25878_summary["Isolate_ID"],
        gse25878_summary["Timepoint"]
    )
)

print("\nAll unique isolate IDs:")
print(
    sorted(
        gse25878_summary["Isolate_ID"]
        .dropna()
        .unique()
    )
)

print("\nSample titles:")
display(
    gse25878_summary[
        ["GSM_ID", "Title", "Isolate_ID",
         "Resistance_Group", "Timepoint"]
    ].sort_values(["Isolate_ID", "Timepoint"])
)

Isolates by resistance group:


Resistance_Group,ART-sensitive,All
Isolate_ID,,
BMT061,10,10
BMT076,9,9
BMT077,9,9
All,28,28



Timepoints available for each isolate:


Timepoint,0,12,16,18,2,24,30,32,36,4,42,48,6,8
Isolate_ID,,,,,,,,,,,,,,
BMT061,1,1,1,0,1,1,0,1,0,1,1,0,1,1
BMT076,1,1,0,1,0,1,1,0,1,0,1,1,1,0
BMT077,1,1,0,1,0,1,1,0,1,0,1,1,1,0



All unique isolate IDs:
['BMT061', 'BMT076', 'BMT077']

Sample titles:


,GSM_ID,Title,Isolate_ID,Resistance_Group,Timepoint
0,GSM635264,ex-vivo_ ART-sensitive_ Xepon Laos_BMT061_0H,BMT061,ART-sensitive,0
5,GSM635269,ex-vivo_ ART-sensitive_ Xepon Laos_BMT061_12H,BMT061,ART-sensitive,12
6,GSM635270,ex-vivo_ ART-sensitive_ Xepon Laos_BMT061_16H,BMT061,ART-sensitive,16
1,GSM635265,ex-vivo_ ART-sensitive_ Xepon Laos_BMT061_2H,BMT061,ART-sensitive,2
7,GSM635271,ex-vivo_ ART-sensitive_ Xepon Laos_BMT061_24H,BMT061,ART-sensitive,24
...,...,...,...,...,...
83,GSM635347,ex-vivo_ ART-resistant_ Pailin Cambodia_CP040_6H,NaN,ART-resistant,6
38,GSM635302,ex-vivo_ ART-sensitive_ Mae Sot Thailand_NHP20...,NaN,ART-sensitive,8
44,GSM635308,ex-vivo_ ART-sensitive_ Mae Sot Thailand_NHP44...,NaN,ART-sensitive,8
50,GSM635314,ex-vivo_ ART-sensitive_ Mae Sot Thailand_NHP44...,NaN,ART-sensitive,8


In [17]:
# Identify samples where isolate ID extraction failed

missing_isolates = gse25878_summary[
    gse25878_summary["Isolate_ID"].isna()
]

print("Samples without extracted isolate ID:",
      len(missing_isolates))

print("\nExamples of unclassified sample titles:")
display(
    missing_isolates[
        ["GSM_ID", "Title", "Resistance_Group", "Timepoint"]
    ].head(30)
)

print("\nResistance groups among unclassified samples:")
display(missing_isolates["Resistance_Group"].value_counts(dropna=False))

Samples without extracted isolate ID: 63

Examples of unclassified sample titles:


,GSM_ID,Title,Resistance_Group,Timepoint
28,GSM635292,ex-vivo_ ART-sensitive_ Xepon Laos_XPN003_0H,ART-sensitive,0
29,GSM635293,ex-vivo_ ART-sensitive_ Xepon Laos_XPN003_6H,ART-sensitive,6
30,GSM635294,ex-vivo_ ART-sensitive_ Xepon Laos_XPN003_12H,ART-sensitive,12
31,GSM635295,ex-vivo_ ART-sensitive_ Xepon Laos_XPN003_18H,ART-sensitive,18
32,GSM635296,ex-vivo_ ART-sensitive_ Xepon Laos_XPN003_24H,ART-sensitive,24
33,GSM635297,ex-vivo_ ART-sensitive_ Xepon Laos_XPN003_30H,ART-sensitive,30
34,GSM635298,ex-vivo_ ART-sensitive_ Xepon Laos_XPN003_36H,ART-sensitive,36
35,GSM635299,ex-vivo_ ART-sensitive_ Xepon Laos_XPN003_42H,ART-sensitive,42
36,GSM635300,ex-vivo_ ART-sensitive_ Xepon Laos_XPN003_48H,ART-sensitive,48
37,GSM635301,ex-vivo_ ART-sensitive_ Mae Sot Thailand_NHP20...,ART-sensitive,0



Resistance groups among unclassified samples:


,count
Resistance_Group,
ART-sensitive,36
ART-resistant,27


In [18]:
import re

# Extract isolate IDs from all sample titles
gse25878_summary["Isolate_ID"] = (
    gse25878_summary["Title"]
    .str.extract(r"_([A-Za-z]+\d+)_\d+H$", expand=False)
)

print("Total samples:", len(gse25878_summary))

print("Samples with identified isolates:",
      gse25878_summary["Isolate_ID"].notna().sum())

print("Unique isolates:",
      gse25878_summary["Isolate_ID"].nunique())

print("\nSamples per isolate and resistance group:")

display(
    pd.crosstab(
        gse25878_summary["Isolate_ID"],
        gse25878_summary["Resistance_Group"],
        margins=True
    )
)

print("\nAll isolate IDs:")
print(sorted(gse25878_summary["Isolate_ID"].dropna().unique()))

Total samples: 91
Samples with identified isolates: 91
Unique isolates: 11

Samples per isolate and resistance group:


Resistance_Group,ART-resistant,ART-sensitive,All
Isolate_ID,,,
BMT061,0,10,10
BMT076,0,9,9
BMT077,0,9,9
CP022,0,9,9
CP025,9,0,9
CP037,9,0,9
CP040,9,0,9
NHP2094,0,6,6
NHP4459,0,6,6



All isolate IDs:
['BMT061', 'BMT076', 'BMT077', 'CP022', 'CP025', 'CP037', 'CP040', 'NHP2094', 'NHP4459', 'NHP4460', 'XPN003']


In [19]:
# Check expression data availability in GSE25878

loaded_samples = 0
empty_samples = 0
sample_shapes = []

for gsm_id, gsm in gse25878.gsms.items():

    if not gsm.table.empty:
        loaded_samples += 1
        sample_shapes.append(gsm.table.shape)
    else:
        empty_samples += 1

print("Total samples:", len(gse25878.gsms))
print("Samples with expression tables:", loaded_samples)
print("Samples with empty tables:", empty_samples)

print("\nExpression table dimensions (first 5 samples):")
print(sample_shapes[:5])

if loaded_samples > 0:
    first_gsm = next(
        gsm for gsm in gse25878.gsms.values()
        if not gsm.table.empty
    )

    print("\nExample expression data:")
    display(first_gsm.table.head())

Total samples: 91
Samples with expression tables: 0
Samples with empty tables: 91

Expression table dimensions (first 5 samples):
[]


In [20]:
# Check supplementary files associated with GSE25878

import requests

url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE25nnn/GSE25878/suppl/"

response = requests.get(url)

print("HTTP status:", response.status_code)
print("\nSupplementary files:")

from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "html.parser")

files = [
    link.get("href")
    for link in soup.find_all("a")
    if link.get("href") and not link.get("href").startswith("?")
]

for file in files:
    print(file)

HTTP status: 403

Supplementary files:
mailto:webadmin@ncbi.nlm.nih.gov
/


In [21]:
import requests
import tarfile
import os

# Download the supplementary archive
url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE25nnn/GSE25878/suppl/GSE25878_RAW.tar"

os.makedirs("data/raw", exist_ok=True)

archive_path = "data/raw/GSE25878_RAW.tar"

if not os.path.exists(archive_path):
    response = requests.get(url, stream=True)
    response.raise_for_status()

    with open(archive_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

print("Archive downloaded.")
print("File size (MB):", round(os.path.getsize(archive_path) / (1024**2), 2))

# List archive contents without extracting
with tarfile.open(archive_path, "r") as tar:
    file_names = tar.getnames()

print("\nNumber of files:", len(file_names))
print("\nFirst 20 files:")

for name in file_names[:20]:
    print(name)

Archive downloaded.
File size (MB): 85.6

Number of files: 91

First 20 files:
GSM635264.gpr.gz
GSM635265.gpr.gz
GSM635266.gpr.gz
GSM635267.gpr.gz
GSM635268.gpr.gz
GSM635269.gpr.gz
GSM635270.gpr.gz
GSM635271.gpr.gz
GSM635272.gpr.gz
GSM635273.gpr.gz
GSM635274.gpr.gz
GSM635275.gpr.gz
GSM635276.gpr.gz
GSM635277.gpr.gz
GSM635278.gpr.gz
GSM635279.gpr.gz
GSM635280.gpr.gz
GSM635281.gpr.gz
GSM635282.gpr.gz
GSM635283.gpr.gz


In [22]:
import tarfile
import gzip

archive_path = "data/raw/GSE25878_RAW.tar"

with tarfile.open(archive_path, "r") as tar:
    first_file = next(
        name for name in tar.getnames()
        if name.endswith(".gpr.gz")
    )

    extracted_file = tar.extractfile(first_file)

    with gzip.open(extracted_file, "rt", errors="replace") as f:
        for i in range(35):
            line = f.readline()
            if not line:
                break
            print(line.rstrip())

ATF	1.0
26	82
"Type=GenePix Results 2"
"DateTime=2009/04/16 17:10:16"
"Settings=D:\STUDENT\MOKS0007\Desktop\Xepon003 Isolate\XEPON003 TP9.gps"
"GalFile=D:\STUDENT\MOKS0007\Desktop\programs & data analysis files in use!!\Microarray Analysis Templates\Microarray Chip Grid\PfZB.32pin.26Nov2008.gal"
"PixelSize=10"
"Wavelengths=635	532"
"ImageFiles=D:\STUDENT\MOKS0007\Desktop\BMT061 isolate\BMT061 TP1 Cy5 vs 3d7 Cy3 Batch 2_160409_635.tif 0	D:\STUDENT\MOKS0007\Desktop\BMT061 isolate\BMT061 TP1 Cy5 vs 3d7 Cy3 Batch 2_160409_532.tif 0"
"NormalizationMethod=None"
"NormalizationFactors=1	1"
"JpegImage=D:\STUDENT\MOKS0007\Desktop\BMT061 isolate\BMT061 TP1 Cy5 vs 3d7 Cy3 Batch 2_160409_0635.jpg"
"RatioFormulations=W1/W2 (635/532)"
"Barcode="
"BackgroundSubtraction=LocalFeature"
"ImageOrigin=1720, 9320"
"JpegOrigin=1760, 9560"
"Creator=GenePix Pro 6.0.1.00"
"Scanner=GenePix 4000B [111537]"
"FocusPosition=0"
"Temperature=25.2"
"LinesAveraged=1"
"Comment="
"PMTGain=570	400"
"ScanPower=100	100"
"Lase

In [23]:
import tarfile
import gzip
import pandas as pd
from io import StringIO

archive_path = "data/raw/GSE25878_RAW.tar"

with tarfile.open(archive_path, "r") as tar:
    first_file = next(
        name for name in tar.getnames()
        if name.endswith(".gpr.gz")
    )

    with tar.extractfile(first_file) as raw_file:
        with gzip.open(raw_file, "rt", errors="replace") as f:
            lines = f.readlines()

# Locate the data table header
header_index = next(
    i for i, line in enumerate(lines)
    if line.startswith('"Block"')
)

# Read the GPR data table
gpr_df = pd.read_csv(
    StringIO("".join(lines[header_index:])),
    sep="\t"
)

print("Sample file:", first_file)
print("Number of probes:", len(gpr_df))
print("\nColumn names:")
print(gpr_df.columns.tolist())

print("\nFirst five probe records:")
display(gpr_df.head())

Sample file: GSM635264.gpr.gz
Number of probes: 11552

Column names:
['Block', 'Column', 'Row', 'Name', 'ID', 'X', 'Y', 'Dia.', 'F635 Median', 'F635 Mean', 'F635 SD', 'B635 Median', 'B635 Mean', 'B635 SD', '% > B635+1SD', '% > B635+2SD', 'F635 % Sat.', 'F532 Median', 'F532 Mean', 'F532 SD', 'B532 Median', 'B532 Mean', 'B532 SD', '% > B532+1SD', '% > B532+2SD', 'F532 % Sat.', 'F3 Median', 'F3 Mean', 'F3 SD', 'B3 Median', 'B3 Mean', 'B3 SD', '% > B3+1SD', '% > B3+2SD', 'F3 % Sat.', 'F4 Median', 'F4 Mean', 'F4 SD', 'B4 Median', 'B4 Mean', 'B4 SD', '% > B4+1SD', '% > B4+2SD', 'F4 % Sat.', 'Ratio of Medians (635/532)', 'Ratio of Means (635/532)', 'Median of Ratios (635/532)', 'Mean of Ratios (635/532)', 'Ratios SD (635/532)', 'Rgn Ratio (635/532)', 'Rgn R�', 'Ratio of Medians (Ratio/2)', 'Ratio of Means (Ratio/2)', 'Median of Ratios (Ratio/2)', 'Mean of Ratios (Ratio/2)', 'Ratios SD (Ratio/2)', 'Rgn Ratio (Ratio/2)', 'Rgn R2 (Ratio/2)', 'Ratio of Medians (Ratio/3)', 'Ratio of Means (Ratio/3

,Block,Column,Row,Name,ID,X,Y,Dia.,F635 Median,F635 Mean,...,F635 Median - B635,F532 Median - B532,F3 Median - B3,F4 Median - B4,F635 Mean - B635,F532 Mean - B532,F3 Mean - B3,F4 Mean - B4,Flags,Normalize
0,1,1,1,bDd2var_157,bDd2var_157,1960,9880,90,61,65,...,10,33,0,0,14,36,0,0,0,0
1,1,2,1,"PFL1410c::ABC transporter, (CT family)",aPFL1410c_1,2150,9840,120,1224,1185,...,1172,262,0,0,1133,254,0,0,0,0
2,1,3,1,PF14_0644::hypothetical protein,aPF14_0644_1,2340,9870,90,332,327,...,283,736,0,0,278,719,0,0,0,0
3,1,4,1,"PFF0955c::hypothetical protein, conserved",aPFF0955c_0,2530,9850,160,58,60,...,8,70,0,0,10,70,0,0,0,0
4,1,5,1,"PFI0755c::6-phosphofructokinase, putative",aPFI0755c_1,2700,9870,90,528,520,...,477,911,0,0,469,910,0,0,0,0


In [24]:
import tarfile
import gzip
import pandas as pd

archive_path = "data/raw/GSE25878_RAW.tar"

file_checks = []

with tarfile.open(archive_path, "r") as tar:

    for name in tar.getnames():

        if not name.endswith(".gpr.gz"):
            continue

        with tar.extractfile(name) as raw_file:
            with gzip.open(raw_file, "rt", errors="replace") as f:
                lines = f.readlines()

        header_index = next(
            i for i, line in enumerate(lines)
            if line.startswith('"Block"')
        )

        probe_count = len(lines) - header_index - 1

        file_checks.append({
            "File": name,
            "Probe_Count": probe_count,
            "Has_F635": any("F635 Median" in x for x in lines[header_index].split("\t")),
            "Has_F532": any("F532 Median" in x for x in lines[header_index].split("\t"))
        })

file_check_df = pd.DataFrame(file_checks)

display(file_check_df.head())

print("\nTotal files:", len(file_check_df))
print("Unique probe counts:", file_check_df["Probe_Count"].nunique())
print("Files with F635:", file_check_df["Has_F635"].sum())
print("Files with F532:", file_check_df["Has_F532"].sum())

,File,Probe_Count,Has_F635,Has_F532
0,GSM635264.gpr.gz,11552,True,True
1,GSM635265.gpr.gz,11552,True,True
2,GSM635266.gpr.gz,11552,True,True
3,GSM635267.gpr.gz,11552,True,True
4,GSM635268.gpr.gz,11552,True,True



Total files: 91
Unique probe counts: 1
Files with F635: 91
Files with F532: 91


In [25]:
import gzip
import os
import pandas as pd

probe_sets = {}

for filename in sorted(os.listdir(gpr_folder)):
    if not filename.endswith(".gpr.gz"):
        continue

    filepath = os.path.join(gpr_folder, filename)

    with gzip.open(filepath, "rt", errors="replace") as f:
        lines = f.readlines()

    header_index = next(
        i for i, line in enumerate(lines)
        if line.startswith('"Block"')
    )

    df = pd.read_csv(
        filepath,
        sep="\t",
        compression="gzip",
        skiprows=header_index,
        low_memory=False
    )

    probe_sets[filename] = set(df["ID"].astype(str))

reference = next(iter(probe_sets.values()))

same_probes = all(
    probes == reference
    for probes in probe_sets.values()
)

print("Files checked:", len(probe_sets))
print("Same probe IDs in every file:", same_probes)
print("Unique probe IDs in first file:", len(reference))

NameError: name 'gpr_folder' is not defined

In [26]:
import os

# Search for the extracted GPR files
gpr_files = []

for root, dirs, files in os.walk("/content"):
    for filename in files:
        if filename.endswith(".gpr.gz"):
            gpr_files.append(os.path.join(root, filename))

print("GPR files found:", len(gpr_files))

if gpr_files:
    gpr_folder = os.path.dirname(gpr_files[0])
    print("GPR folder:", gpr_folder)
    print("Example files:")
    for file in gpr_files[:5]:
        print(os.path.basename(file))
else:
    print("No GPR files found. The archive may need to be extracted again.")

GPR files found: 0
No GPR files found. The archive may need to be extracted again.


In [27]:
import os

print("Files in /content:")
for item in os.listdir("/content"):
    path = os.path.join("/content", item)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"{item} — {size_mb:.2f} MB")
    else:
        print(f"{item}/")

Files in /content:
.config/
data/
sample_data/


In [28]:
import os

for root, dirs, files in os.walk("/content/data"):
    print(f"\nFolder: {root}")
    for filename in files[:20]:
        path = os.path.join(root, filename)
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"  {filename} — {size_mb:.2f} MB")


Folder: /content/data

Folder: /content/data/raw
  GSE25878.txt — 0.35 MB
  GSE25878_RAW.tar — 85.60 MB


In [29]:
import os
import tarfile

archive_path = "/content/data/raw/GSE25878_RAW.tar"
gpr_folder = "/content/data/raw/GSE25878_GPR"

os.makedirs(gpr_folder, exist_ok=True)

with tarfile.open(archive_path, "r") as tar:
    tar.extractall(path=gpr_folder)

gpr_files = [
    f for f in os.listdir(gpr_folder)
    if f.endswith(".gpr.gz")
]

print("Extracted GPR files:", len(gpr_files))
print("GPR folder:", gpr_folder)
print("First five files:", gpr_files[:5])

Extracted GPR files: 91
GPR folder: /content/data/raw/GSE25878_GPR
First five files: ['GSM635334.gpr.gz', 'GSM635273.gpr.gz', 'GSM635341.gpr.gz', 'GSM635280.gpr.gz', 'GSM635323.gpr.gz']


/tmp/ipykernel_6705/2968586642.py:10: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=gpr_folder)


In [32]:
import gzip
import pandas as pd
import os

probe_sets = {}
failed_files = []

for filename in sorted(os.listdir(gpr_folder)):

    if not filename.endswith(".gpr.gz"):
        continue

    filepath = os.path.join(gpr_folder, filename)

    try:
        with gzip.open(filepath, "rt", encoding="latin1") as f:
            lines = f.readlines()

        header_index = next(
            i for i, line in enumerate(lines)
            if line.startswith('"Block"')
        )

        df = pd.read_csv(
            filepath,
            sep="\t",
            compression="gzip",
            skiprows=header_index,
            encoding="latin1",
            low_memory=False
        )

        probe_sets[filename] = set(df["ID"].astype(str))

    except Exception as e:
        failed_files.append((filename, str(e)))

print("Files successfully checked:", len(probe_sets))
print("Files with errors:", len(failed_files))

if probe_sets:
    reference = next(iter(probe_sets.values()))

    same_probes = all(
        probes == reference
        for probes in probe_sets.values()
    )

    print("Same probe IDs in every file:", same_probes)
    print("Unique probe IDs in first file:", len(reference))

if failed_files:
    print("\nFailed files:")
    for item in failed_files:
        print(item)

Files successfully checked: 91
Files with errors: 0
Same probe IDs in every file: True
Unique probe IDs in first file: 11426
